# FT05 no Colab Pro — classificador sintético

Abra este notebook no Colab com a conta que tem Pro. A conta GitHub pode ser diferente: o repositório é público. Esta entrega prepara a execução, mas **não treinou** nem mediu GPU. A fixture de 24 registros não é corpus aprovado. Execute as células em ordem, reveja cada operação de rede e escolha apenas dados sintéticos com rótulos e grupos revisados. O Colab pode interromper a VM, e Pro não garante uma GPU específica.


In [ ]:
from pathlib import Path
import subprocess, sys
print('Python:', sys.version.split()[0])
probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True) if Path('/usr/bin/nvidia-smi').exists() else None
print('GPU observada:', probe.stdout.splitlines()[0] if probe and probe.returncode == 0 else 'não detectada')
print('Saldo/unidades e tipo de runtime: confira na interface Colab antes do treino.')


## 1. Código versionado (operação de rede explícita)

Copie o SHA de 40 caracteres da PR empilhada `feat/ft05-colab-pro` para `EXPECTED_REPO_SHA`. O clone abaixo só ocorre após `FETCH_SOURCE=True`, e o notebook recusa um branch que tenha avançado. Inspecione os arquivos `ft05.py` e `train.py` antes de executar a etapa de modelo.


In [ ]:
import re
FETCH_SOURCE = False
EXPECTED_REPO_SHA = ''  # SHA da PR, não o SHA do modelo
REPO_URL = 'https://github.com/ofelipepeixoto/radar-model-router.git'
REPO_BRANCH = 'feat/ft05-colab-pro'
REPO_DIR = Path('/content/radar-model-router-ft05')
def verified_source():
    assert re.fullmatch(r'[0-9a-f]{40}', EXPECTED_REPO_SHA), 'Informe o SHA da PR'
    assert REPO_DIR.is_dir(), 'Clone o repositório primeiro'
    actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == EXPECTED_REPO_SHA, f'Código mudou: {actual}; inspecione a PR'
    return actual
if FETCH_SOURCE:
    assert re.fullmatch(r'[0-9a-f]{40}', EXPECTED_REPO_SHA), 'Informe o commit de 40 caracteres'
    assert not REPO_DIR.exists(), 'Diretório já existe: não sobrescrever código'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    print('Código fixado:', verified_source())
else:
    print('Clone desativado; configure o SHA e ative após a revisão.')


## 2. Ensaio offline, sem GPU

Esta etapa valida a fixture fictícia e mostra o plano. Ela não consulta modelo nem configura a conta Google.


In [ ]:
if REPO_DIR.is_dir():
    verified_source()
    FT05_DIR = REPO_DIR / 'experiments' / 'ft05'
    sys.path.insert(0, str(FT05_DIR))
    from train import run
    from ft05 import load_frozen, prepare, target
    print(run(FT05_DIR / 'synthetic.jsonl', FT05_DIR / 'training-config.json',
              '/content/unused-plan', execute=False))
else:
    print('Execute primeiro o clone versionado.')


## 3. Persistência e corpus revisado

Se optar pelo Drive desta conta Pro, monte-o apenas depois de revisar o notebook. Defina `PERSIST_ROOT` como uma pasta exclusiva deste FT05 em `/content/drive/MyDrive/` e `DATASET_PATH` para o JSONL sintético revisado (200–500 registros). Não use dados reais/privados. O diretório `frozen` registra train/validation/test; preserve-o como artefato e nunca ajuste decisões pelos casos de teste. A flag `reviewed: true` é declaração do responsável, não revisão automática.


In [ ]:
MOUNT_DRIVE = False
PERSIST_ROOT = ''  # exemplo: /content/drive/MyDrive/ft05/run-2026-10
DATASET_PATH = ''   # caminho do JSONL sintético revisado, dentro do Drive
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    assert PERSIST_ROOT.startswith('/content/drive/MyDrive/'), 'Use uma pasta exclusiva no Drive'
    root = Path(PERSIST_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    print('Destino:', root)
else:
    print('Drive não montado. Ensaio offline disponível; treino permanece bloqueado.')


In [ ]:
FREEZE_REVIEWED = False
if FREEZE_REVIEWED:
    assert MOUNT_DRIVE and REPO_DIR.is_dir() and DATASET_PATH, 'Informe código, Drive e corpus revisado'
    dataset = Path(DATASET_PATH)
    frozen = Path(PERSIST_ROOT) / 'frozen'
    if not frozen.exists():
        prepare(dataset, frozen, training=True)  # jamais aceita a fixture não revisada
    parts, frozen_sha256 = load_frozen(dataset, frozen)
    from collections import Counter
    for name, records in parts.items():
        print(name, len(records), 'categorias:', dict(Counter(target(r['messages'][-1]['content'])['categoria'] for r in records)))
    print('Hash do manifesto congelado:', frozen_sha256)
    print('Confirme manualmente grupos, representatividade e distribuição de urgência antes do treino.')
else:
    print('Congelamento desativado até concluir revisão do corpus e grupos.')


## 4. Dependências e modelo

Revise compatibilidade no runtime observado e os pacotes da instalação. Preencha versões exatas antes de habilitar `INSTALL_DEPS`; a célula instala apenas esses seis pacotes por PyPI e registra as versões efetivas. A revisão do modelo Qwen está fixada no JSON do repositório. Um ambiente Colab novo pode trazer bibliotecas transitivas diferentes: guarde o lock gerado e reinicie o runtime se a instalação solicitar. Não ligue o treino se a instalação falhar.


In [ ]:
INSTALL_DEPS = False
PINNED_PACKAGES = {  # substitua cada vazio por versão testada e revisada, sem ==
    'unsloth': '', 'torch': '', 'transformers': '',
    'trl': '', 'datasets': '', 'peft': '',
}
if INSTALL_DEPS:
    assert MOUNT_DRIVE and REPO_DIR.is_dir(), 'Revise código e destino persistente primeiro'
    assert set(PINNED_PACKAGES) == {'unsloth','torch','transformers','trl','datasets','peft'}
    assert all(re.fullmatch(r'[0-9][A-Za-z0-9.+_-]*', v) for v in PINNED_PACKAGES.values()), 'Fixe versões exatas'
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    *[f'{name}=={version}' for name, version in PINNED_PACKAGES.items()]], check=True)
    lock = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
    lock_path = Path(PERSIST_ROOT) / 'requirements-lock.txt'
    if lock_path.exists():
        assert lock_path.read_text() == lock, 'Lock anterior difere; crie outra execução'
    else:
        lock_path.write_text(lock)
    print('Ambiente registrado em', lock_path)
else:
    print('Instalação desativada. Consulte README e documentação oficial da Unsloth/TRL.')


## 5. Modelo original → treino LoRA → modelo personalizado

O script aplica exatamente as mesmas perguntas do teste congelado antes e depois do treino; a resposta esperada nunca entra no prompt de inferência. Apenas treino e validação entram no `SFTTrainer`. O output inclui previsões brutas, métricas, comparação por ID, adaptador e hashes. O teste é consumido para avaliação uma única vez nesta decisão; não escolha hiperparâmetros nem reescreva o corpus olhando seus erros. O início do treino pode consumir unidades Pro.


In [ ]:
RUN_TRAINING = False
BUDGET_CONFIRMED = False
DEPENDENCIES_REVIEWED = False
RUN_NAME = 'run-001'  # novo nome para nova tentativa, sem sobrescrever resultados
if RUN_TRAINING:
    assert MOUNT_DRIVE and FREEZE_REVIEWED and INSTALL_DEPS
    assert BUDGET_CONFIRMED and DEPENDENCIES_REVIEWED
    import importlib.metadata
    from ft05 import strict_json
    from train import check_config
    config = strict_json((FT05_DIR / 'training-config.json').read_text())
    config['gpu_budget_approved'] = BUDGET_CONFIRMED
    config['dependency_reviewed'] = DEPENDENCIES_REVIEWED
    config['package_versions'] = {name: importlib.metadata.version(name) for name in PINNED_PACKAGES}
    assert config['package_versions'] == PINNED_PACKAGES, 'Versões divergem das escolhidas'
    check_config(config)
    config_path = Path(PERSIST_ROOT) / f'{RUN_NAME}-config.json'
    output = Path(PERSIST_ROOT) / RUN_NAME
    assert not config_path.exists() and not output.exists(), 'Execução existente; escolha novo nome'
    config_path.write_text(json.dumps(config, indent=2) + '\n')
    import train
    result = train.run(dataset, config_path, output, execute=True, frozen_dir=frozen)
    print('Treino e avaliação concluídos:', result)
else:
    print('Treino desativado. Verifique todos os gates e o holdout antes de ativar.')


## 6. Ler a comparação

Abra `comparison.json` e confira o delta de acerto exato, IDs melhorados e IDs regredidos. `model-baseline.json` e `model-finetuned.json` contêm métricas; `predictions-*` contêm saídas brutas. Se a célula de treino parou, não interprete arquivos parciais como resultado final. Preserve o diretório e o commit usados; não envie pesos/artefatos a outros projetos sem outra decisão.


In [ ]:
if RUN_TRAINING and (output / 'run.json').exists():
    import json
    report = json.loads((output / 'comparison.json').read_text())
    print({k: report[k] for k in ('n','delta_exact_match','delta_category_macro_f1','delta_urgency_macro_f1','delta_valid_json_rate','improved_ids','regressed_ids')})
    print('Original:', report['original'])
    print('Personalizado:', report['personalized'])
else:
    print('Ainda não há comparação de modelo executada.')
